In [1]:
import pandas as pd 
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder,RobustScaler,QuantileTransformer

In [2]:
# === 1. Load dataset ===
url = "https://raw.githubusercontent.com/dicodingacademy/dicoding_dataset/main/employee/employee_data.csv"
dataset = pd.read_csv(url)
dataset.head().T

,0,1,2,3,4
EmployeeId,1,2,3,4,5
Age,38,37,51,42,40
Attrition,NaN,1.0,1.0,0.0,NaN
BusinessTravel,Travel_Frequently,Travel_Rarely,Travel_Rarely,Travel_Frequently,Travel_Rarely
DailyRate,1444,1141,1323,555,1194
Department,Human Resources,Research & Development,Research & Development,Sales,Research & Development
DistanceFromHome,1,11,4,26,2
Education,4,2,4,3,4
EducationField,Other,Medical,Life Sciences,Marketing,Medical
EmployeeCount,1,1,1,1,1


In [4]:
model_df = dataset.copy()
model_df = model_df.drop(['EmployeeId'], axis=1)

# === 2. Label Encoding ===
le_dict = {}

for col in model_df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    model_df[col] = le.fit_transform(model_df[col].astype(str))
    le_dict[col] = le

# === 3. Pisahkan data ===
labeled_df = model_df[model_df['Attrition'].notna()]
labeled_df = model_df.loc[labeled_df.index]

unlabeled_df = model_df[model_df['Attrition'].isna()]
unlabeled_df = model_df.loc[unlabeled_df.index]

X = labeled_df.drop('Attrition', axis=1)
y = labeled_df['Attrition']
num_cols = X.select_dtypes(include=[np.number]).columns

# === 4. Scaling dan Normalisasi ===
scaler = RobustScaler()
X_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_array, columns=X.columns)

quantile_transformer = QuantileTransformer(output_distribution='normal')
quantile_transformer.fit(X_scaled)
X_scaled_df = pd.DataFrame(quantile_transformer.transform(X_scaled), columns=X.columns)

# === 5. Preprocess unlabeled ===
for col in unlabeled_df.select_dtypes(include='object').columns:
    le = le_dict[col]
    unlabeled_df[col] = le.transform(unlabeled_df[col].astype(str))

X_unlabeled = unlabeled_df.drop('Attrition', axis=1)
X_unlabeled_scaled_df = pd.DataFrame(scaler.transform(X_unlabeled), columns=X_unlabeled.columns)
X_unlabeled_scaled_df = pd.DataFrame(quantile_transformer.transform(X_unlabeled_scaled_df), columns=X_unlabeled.columns)

In [5]:
# === 6. Load Model & Prediksi ===
model = joblib.load("best_randomForrest_model.pkl")

predicted_attrition_labeled = model.predict(X_scaled_df)
predicted_attrition_unlabeled = model.predict(X_unlabeled_scaled_df)
unlabeled_df['Predicted_Attrition'] = predicted_attrition_unlabeled
labeled_df['Predicted_Attrition'] = predicted_attrition_labeled

In [10]:
# Fungsi bantu untuk konversi kolom numerik jadi int (tanpa mengubah kolom float murni seperti income)
def convert_numeric_to_int(df, original_df):
    for col in df.columns:
        if np.issubdtype(original_df[col].dtype, np.integer):
            df[col] = df[col].round().astype(int)
    return df

# === UNLABELED ===
X_inverse_quantile_unlabeled = quantile_transformer.inverse_transform(X_unlabeled_scaled_df)
X_inverse_unlabeled = scaler.inverse_transform(X_inverse_quantile_unlabeled)
X_inverse_unlabeled_df = pd.DataFrame(X_inverse_unlabeled, columns=X_unlabeled.columns)

# Konversi nilai numerik ke integer
X_inverse_unlabeled_df = convert_numeric_to_int(X_inverse_unlabeled_df, dataset.loc[unlabeled_df.index])

# Decode kembali kolom kategorikal
for col in le_dict:
    if col in X_inverse_unlabeled_df.columns:
        le = le_dict[col]
        max_class = len(le.classes_) - 1
        X_inverse_unlabeled_df[col] = X_inverse_unlabeled_df[col].clip(0, max_class)
        X_inverse_unlabeled_df[col] = le.inverse_transform(X_inverse_unlabeled_df[col].astype(int))

# Tambahkan kolom Attrition asli dan prediksi
X_inverse_unlabeled_df['Attrition'] = dataset.loc[unlabeled_df.index, 'Attrition'].values
X_inverse_unlabeled_df['Predicted_Attrition'] = predicted_attrition_unlabeled
X_inverse_unlabeled_df.index = dataset.loc[unlabeled_df.index].index
display(X_inverse_unlabeled_df)

# === LABELED ===
X_inverse_quantile_labeled = quantile_transformer.inverse_transform(X_scaled_df)
X_inverse_labeled = scaler.inverse_transform(X_inverse_quantile_labeled)
X_inverse_labeled_df = pd.DataFrame(X_inverse_labeled, columns=X.columns)

# Konversi nilai numerik ke integer
X_inverse_labeled_df = convert_numeric_to_int(X_inverse_labeled_df, dataset.loc[labeled_df.index])

# Decode kembali kolom kategorikal
for col in le_dict:
    if col in X_inverse_labeled_df.columns:
        le = le_dict[col]
        max_class = len(le.classes_) - 1
        X_inverse_labeled_df[col] = X_inverse_labeled_df[col].clip(0, max_class)
        X_inverse_labeled_df[col] = le.inverse_transform(X_inverse_labeled_df[col].astype(int))

# Tambahkan kolom Attrition asli dan prediksi
X_inverse_labeled_df['Attrition'] = dataset.loc[labeled_df.index, 'Attrition'].values
X_inverse_labeled_df['Predicted_Attrition'] = predicted_attrition_labeled
X_inverse_labeled_df.index = dataset.loc[labeled_df.index].index
display(X_inverse_labeled_df)


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,Predicted_Attrition
0,38,Travel_Frequently,1444,Human Resources,1,4,Other,1,4,Male,...,1,7,2,3,6,2,1,2,NaN,0.0
4,40,Travel_Rarely,1194,Research & Development,2,4,Medical,1,3,Female,...,3,20,2,3,5,3,0,2,NaN,0.0
5,29,Travel_Rarely,352,Human Resources,6,1,Medical,1,4,Male,...,0,1,3,3,1,0,0,0,NaN,0.0
12,47,Travel_Rarely,571,Sales,14,3,Medical,1,3,Female,...,1,11,4,2,5,4,1,2,NaN,0.0
18,25,Travel_Frequently,772,Research & Development,2,1,Life Sciences,1,4,Male,...,2,7,6,3,7,7,0,7,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1443,24,Travel_Frequently,567,Research & Development,2,1,Technical Degree,1,1,Female,...,0,6,2,3,6,3,1,3,NaN,1.0
1447,42,Travel_Frequently,288,Research & Development,2,3,Life Sciences,1,4,Male,...,1,24,3,1,20,8,13,9,NaN,0.0
1448,38,Travel_Rarely,437,Sales,16,3,Life Sciences,1,2,Female,...,0,8,5,4,3,2,1,2,NaN,0.0
1462,41,Travel_Rarely,1206,Sales,23,2,Life Sciences,1,4,Male,...,0,21,2,3,2,0,0,2,NaN,0.0


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EnvironmentSatisfaction,Gender,...,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,Predicted_Attrition
1,36,Travel_Rarely,1068,Research & Development,9,2,Medical,1,1,Female,...,0,13,2,1,3,2,0,2,1.0,1.0
2,50,Travel_Rarely,1185,Research & Development,4,3,Life Sciences,1,1,Male,...,3,17,2,3,10,2,1,7,1.0,1.0
3,41,Travel_Frequently,617,Sales,26,3,Marketing,1,3,Female,...,1,23,2,3,33,4,5,8,0.0,0.0
6,39,Travel_Rarely,1053,Sales,3,2,Medical,1,2,Male,...,3,7,2,2,5,3,0,2,0.0,0.0
7,53,Travel_Rarely,721,Research & Development,3,3,Medical,1,3,Male,...,1,25,2,3,5,2,1,4,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1464,31,Non-Travel,1217,Research & Development,25,2,Technical Degree,1,2,Male,...,0,10,2,2,10,7,1,8,1.0,1.0
1465,37,Travel_Rarely,342,Research & Development,3,3,Life Sciences,1,3,Female,...,0,10,4,3,3,2,0,2,0.0,0.0
1467,31,Travel_Rarely,1276,Research & Development,10,2,Life Sciences,1,3,Female,...,0,5,4,2,3,2,0,2,1.0,1.0
1468,39,Non-Travel,541,Research & Development,16,2,Life Sciences,1,3,Male,...,1,7,0,3,5,2,0,2,0.0,0.0


In [11]:
combined_df = pd.concat([X_inverse_labeled_df, X_inverse_unlabeled_df]).sort_index()
combined_df['EmployeeId'] = dataset['EmployeeId'].values
cols = combined_df.columns.tolist()
cols.insert(cols.index('Age'), cols.pop(cols.index('EmployeeId')))
combined_df = combined_df[cols]
display(combined_df.head().T)

,0,1,2,3,4
EmployeeId,1,2,3,4,5
Age,38,36,50,41,40
BusinessTravel,Travel_Frequently,Travel_Rarely,Travel_Rarely,Travel_Frequently,Travel_Rarely
DailyRate,1444,1068,1185,617,1194
Department,Human Resources,Research & Development,Research & Development,Sales,Research & Development
DistanceFromHome,1,9,4,26,2
Education,4,2,3,3,4
EducationField,Other,Medical,Life Sciences,Marketing,Medical
EmployeeCount,1,1,1,1,1
EnvironmentSatisfaction,4,1,1,3,3
